In [1]:
import pandas as pd
import numpy as np
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter('ignore')

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import PolynomialFeatures

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import copy
import shap 

In [5]:
df = pd.read_csv("C:/Users/Nayan/Downloads/F1Stop.csv")
df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [7]:
df = df.drop('id', axis = 1)
df

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439135,D755,MEDIUM,Miami Grand Prix,2023,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0
439136,D731,MEDIUM,Miami Grand Prix,2023,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0
439137,D716,MEDIUM,Miami Grand Prix,2023,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0
439138,D665,HARD,Abu Dhabi Grand Prix,2023,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0


In [8]:
df['Compound_X_Race'] = df['Compound']+'_'+df['Race']
df['Compound_X_Year'] = df['Compound']+'_'+df['Year'].astype(str)
df['Race_X_Year'] = df['Race']+"_"+df.Year.astype(str)
df.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Compound_X_Race,Compound_X_Year,Race_X_Year
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,HARD_Canadian Grand Prix,HARD_2022,Canadian Grand Prix_2022
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0,HARD_Dutch Grand Prix,HARD_2025,Dutch Grand Prix_2025
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,HARD_Austrian Grand Prix,HARD_2022,Austrian Grand Prix_2022
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,MEDIUM_Pre-Season Testing,MEDIUM_2023,Pre-Season Testing_2023
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,HARD_Azerbaijan Grand Prix,HARD_2022,Azerbaijan Grand Prix_2022


In [10]:
cat_col = df.select_dtypes(include='object').columns
num_col = df.select_dtypes(exclude='object').columns

In [ ]:
def cnt_encode(df1, col_target):
    for col in cat_col:
        df2 = df1[col].value_counts().reset_index()
        df2['rate'] = df2['count']/df2['count'].max()
        df2.columns = [col,'cnt_'+col, 'rate_'+col]
        df2 = pd.merge(df1, df2, how='left', on = col)
        del df2['Driver']
    return df2

In [13]:
ss = cnt_encode(df, cat_col)
ss.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Compound_X_Race,Compound_X_Year,Race_X_Year,cnt_Race_X_Year,rate_Race_X_Year
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,HARD_Canadian Grand Prix,HARD_2022,Canadian Grand Prix_2022,5016,0.634294
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0,HARD_Dutch Grand Prix,HARD_2025,Dutch Grand Prix_2025,6416,0.811330
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,HARD_Austrian Grand Prix,HARD_2022,Austrian Grand Prix_2022,4534,0.573343
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,MEDIUM_Pre-Season Testing,MEDIUM_2023,Pre-Season Testing_2023,7855,0.993298
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,HARD_Azerbaijan Grand Prix,HARD_2022,Azerbaijan Grand Prix_2022,2564,0.324229


In [16]:
df.head()

,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Compound_X_Race,Compound_X_Year,Race_X_Year
0,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,HARD_Canadian Grand Prix,HARD_2022,Canadian Grand Prix_2022
1,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0,HARD_Dutch Grand Prix,HARD_2025,Dutch Grand Prix_2025
2,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,HARD_Austrian Grand Prix,HARD_2022,Austrian Grand Prix_2022
3,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,MEDIUM_Pre-Season Testing,MEDIUM_2023,Pre-Season Testing_2023
4,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,HARD_Azerbaijan Grand Prix,HARD_2022,Azerbaijan Grand Prix_2022


In [17]:
col_target = df.columns[-1]
col_target

'Race_X_Year'

In [20]:
def target_encoding(df_train, col_target):
    
    for col in cat_col:
        df_cat_te = df_train.groupby([col], as_index=False, dropna=False)[col_target].agg(['mean', 'std'])
        df_cat_te.columns = [col, 'TE_mean_' + col, 'TE_std_' + col]

        df_train = pd.merge(df_train, df_cat_te, how = 'left', on = col)

        del df_train[col]
        
    return df_train

In [21]:
ss1 = target_encoding(df, col_target)
ss1.head()

KeyError: 'Driver'